# Notebook 6: Downstream Analysis - Alpha Diversity
In this notebook, we will compute and visualize **Alpha Diversity** metrics using our `phyloseq_final.rds` object. Alpha diversity measures the richness and evenness of bacterial communities within individual samples.

In [1]:
import os
import subprocess

project_dir = "/home/azureuser/Microbiome_project"
container_path = os.path.join(project_dir, "dada2.sif")

r_code = """
# 1. Load local library paths and libraries
local_lib <- "/home/azureuser/Microbiome_project/R_libs"
.libPaths(c(local_lib, .libPaths()))

library(phyloseq)
library(tidyverse)
library(ggplot2)

# 2. Load the phyloseq object
cat("Loading phyloseq object...\n")
ps <- readRDS("phyloseq_final.rds")

# 3. Calculate Alpha Diversity
cat("Computing alpha diversity metrics...\n")
alpha_df <- estimate_richness(ps, measures = c("Observed", "Shannon"))

# Combine alpha diversity with sample metadata
meta_df <- as(sample_data(ps), "data.frame")
alpha_combined <- cbind(alpha_df, meta_df)

# 4. Preview the first few rows of calculated diversity
print(head(alpha_combined[, c("Observed", "Shannon", "Timepoint", "Supplementation")]))

# 5. Generate a boxplot comparing Alpha Diversity across groups (e.g., Supplementation)
# Note: You can change 'Supplementation' to any other valid column name from your metadata
p <- plot_richness(ps, x = "Supplementation", measures = c("Observed", "Shannon"), color = "Supplementation") +
     geom_boxplot(alpha = 0.6) +
     theme_bw() +
     theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
     ggtitle("Alpha Diversity by Supplementation Group")

# Save the plot
ggsave("alpha_diversity_plot.png", plot = p, width = 8, height = 6, dpi = 300)
cat("\nAlpha diversity plot successfully saved to alpha_diversity_plot.png!\n")
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)


R version 4.4.2 (2024-10-31) -- "Pile of Leaves"
Copyright (C) 2024 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> 
> # 1. Load local library paths and libraries
> local_lib <- "/home/azureuser/Microbiome_project/R_libs"
> .libPaths(c(local_lib, .libPaths()))
> 
> library(phyloseq)
> library(tidyverse)
> library(ggplot2)
> 
> # 2. Load the phyloseq object
> cat("Loading phyloseq object...
+ ")
Loading phyloseq object...
> ps <- 

# Notebook 6 (Continued): Beta Diversity Analysis
In this section, we will compute and visualize **Beta Diversity** to evaluate differences in overall microbial community composition between samples. 
We will use:
1. **Bray-Curtis distance** matrix to measure dissimilarity.
2. **PCoA (Principal Coordinate Analysis)** ordination to visualize sample clustering in a 2D space.
3. **PERMANOVA (Adonis)** statistical test to check if the bacterial composition differs significantly between groups (e.g., Supplementation).

In [2]:
import os
import subprocess

project_dir = "/home/azureuser/Microbiome_project"
container_path = os.path.join(project_dir, "dada2.sif")

r_code = """
# 1. Load local library paths and required packages
local_lib <- "/home/azureuser/Microbiome_project/R_libs"
.libPaths(c(local_lib, .libPaths()))

library(phyloseq)
library(tidyverse)
library(vegan)
library(ggplot2)

# 2. Load the phyloseq object
cat("Loading phyloseq object for Beta Diversity...\n")
ps <- readRDS("phyloseq_final.rds")

# 3. Perform Ordination (PCoA using Bray-Curtis distance)
cat("Calculating Bray-Curtis distance and running PCoA ordination...\n")
ps.ord <- ordinate(ps, method = "PCoA", distance = "bray")

# 4. Plot Ordination colored by Supplementation (or change to any other metadata variable)
p_beta <- plot_ordination(ps, ps.ord, color = "Supplementation") +
          geom_point(alpha = 0.7, size = 2) +
          theme_bw() +
          ggtitle("PCoA - Bray-Curtis Distance (Beta Diversity)")

# Save the plot
ggsave("beta_diversity_pcoa.png", plot = p_beta, width = 8, height = 6, dpi = 300)
cat("\nBeta diversity PCoA plot successfully saved to beta_diversity_pcoa.png!\n")

# 5. Run PERMANOVA (Adonis) test to check statistical significance between groups
cat("\nRunning PERMANOVA (Adonis) statistical test...\n")
meta_df <- data.frame(sample_data(ps))
bray_dist <- distance(ps, method = "bray")

# Test effect of Supplementation (Adjust variable name if needed)
permanova_res <- adonis2(bray_dist ~ Supplementation, data = meta_df, permutations = 999)
print(permanova_res)
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)


R version 4.4.2 (2024-10-31) -- "Pile of Leaves"
Copyright (C) 2024 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> 
> # 1. Load local library paths and required packages
> local_lib <- "/home/azureuser/Microbiome_project/R_libs"
> .libPaths(c(local_lib, .libPaths()))
> 
> library(phyloseq)
> library(tidyverse)
> library(vegan)
> library(ggplot2)
> 
> # 2. Load the phyloseq object
> cat("Loading phyloseq object for Beta Diversity

# Taxonomic Composition Analysis
To complement our diversity analyses, we will now examine the relative abundance of bacterial taxa at the **Family** level. This will help us identify which specific bacterial groups dominate the microbiome across our study groups and provide biological context for our conclusions.

In [3]:
import os
import subprocess

project_dir = "/home/azureuser/Microbiome_project"
container_path = os.path.join(project_dir, "dada2.sif")

r_code = """
# 1. Load local library paths and required packages
local_lib <- "/home/azureuser/Microbiome_project/R_libs"
.libPaths(c(local_lib, .libPaths()))

library(phyloseq)
library(tidyverse)
library(ggplot2)

# 2. Load the phyloseq object
cat("Loading phyloseq object for Taxonomic Composition...\n")
ps <- readRDS("phyloseq_final.rds")

# 3. Aggregate taxa to Family level and convert to relative abundance
ps_fam <- tax_glom(ps, taxrank = "Family")
ps_rel <- transform_sample_counts(ps_fam, function(x) x / sum(x))

# 4. Melt phyloseq object into a tidy data frame for ggplot
df <- psmelt(ps_rel)

# 5. Select Top 10 most abundant families and group the rest as 'Other'
top10_families <- names(sort(tapply(df$Abundance, df$Family, sum), decreasing = TRUE))[1:10]
df_top <- df %>% 
  mutate(Family = ifelse(Family %in% top10_families, as.character(Family), "Other"))

# 6. Generate a stacked bar plot grouped by Supplementation
p_bar <- ggplot(df_top, aes(x = Supplementation, y = Abundance, fill = Family)) +
         geom_bar(stat = "identity", position = "fill") +
         theme_bw() +
         labs(title = "Top 10 Bacterial Families by Supplementation Group",
              y = "Relative Abundance", x = "Supplementation Group") +
         theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
         scale_fill_viridis_d(option = "turbo")

# Save the plot
ggsave("taxonomic_barplot.png", plot = p_bar, width = 9, height = 6, dpi = 300)
cat("\nTaxonomic barplot successfully saved to taxonomic_barplot.png!\n")
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)


R version 4.4.2 (2024-10-31) -- "Pile of Leaves"
Copyright (C) 2024 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> 
> # 1. Load local library paths and required packages
> local_lib <- "/home/azureuser/Microbiome_project/R_libs"
> .libPaths(c(local_lib, .libPaths()))
> 
> library(phyloseq)
> library(tidyverse)
> library(ggplot2)
> 
> # 2. Load the phyloseq object
> cat("Loading phyloseq object for Taxonomic Composition...
+ ")
L

# Notebook 6: Summary and Ecological Insights
In this final section of Notebook 6, we synthesize our findings from Alpha diversity, Beta diversity (Bray-Curtis PCoA & PERMANOVA), and Taxonomic composition barplots to evaluate the overall impact of the supplementation on the microbiome structure.